In [1]:
from the_well.data.datasets import TrajectoryData, TrajectoryMetadata
from the_well.data.normalization import ZScoreNormalization
from the_well.data import WellDataset

from the_well.benchmark.metrics.spatial import VRMSE
from typing import Sequence, Dict, Tuple, Optional
from einops import rearrange
from pathlib import Path
import fsspec
import pickle
import torch
import yaml
import sys
import os

from unittest.mock import MagicMock
# Mock torch_harmonics as a proper package with all submodules neuralop touches
mock_th = MagicMock()
mock_th.__path__ = []          # marks it as a package
mock_th.__spec__ = MagicMock() # satisfies importlib checks

sys.modules['torch_harmonics']          = mock_th
sys.modules['torch_harmonics.examples'] = MagicMock()
sys.modules['torch_harmonics.layers']   = MagicMock()
sys.modules['torch_harmonics.grids']    = MagicMock()

from the_well.benchmark.models import UNetClassic, UNetConvNext, FNO, TFNO

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.neural_ops import NeuralOperatorModule

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

/opt/anaconda3/envs/diffusion_env/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# Load Data

In [2]:
well_base_path = f"{PROJECT_ROOT}/data/datasets"

#well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'rayleigh_benard'
well_dataset_name = 'planetswe'

stats_path = f"{well_base_path}/{well_dataset_name}/stats.yaml"

dataset = get_well_dataset(
    path=well_base_path,
    physics=well_dataset_name,
    split='test',
    steps=-1,
    transform = ['log_scalars'],   
    augment = [],
)

with fsspec.open(stats_path, mode="r") as f:
    stats = yaml.safe_load(f)

core_field_names          = dataset.core_field_names
core_constant_field_names = dataset.core_constant_field_names

normalizer = ZScoreNormalization(stats, core_field_names, core_constant_field_names)

# Load Baseline Models

In [3]:
unet_classic  = UNetClassic.from_pretrained(f"polymathic-ai/UNetClassic-{well_dataset_name}")
unet_convnext = UNetConvNext.from_pretrained(f"polymathic-ai/UNetConvNext-{well_dataset_name}")
fno           = FNO.from_pretrained(f"polymathic-ai/FNO-{well_dataset_name}")
#tfno          = TFNO.from_pretrained(f"polymathic-ai/TFNO-{well_dataset_name}")

baselines = {
    'UNetClassic'   : unet_classic,
    'UNetConvNext'  : unet_convnext,
    'FNO'           : fno,
    'TFNO'          : tfno,
}

NameError: name 'tfno' is not defined

In [4]:
class CPUForwardWrapper(torch.nn.Module):
    """
    Makes a CPU-resident model usable in an MPS pipeline.
    Input arrives on any device, forward runs on CPU, output
    is returned on the same device as the input.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model.to("cpu")

    def to(self, device, **kwargs):
        # Intercept .to("mps") — keep inner model on CPU always
        self._output_device = torch.device(device) if isinstance(device, str) else device
        return self  # do NOT call super().to() — that would move inner model

    def forward(self, x):
        out = self.model(x.cpu())   # move input to CPU, run, 
        return out.to(x.device)     # return output to original device (mps)
    
SPECTRAL_MODELS = {"FNO", "TFNO"}
dname = dataset.metadata.dataset_name
device = torch.device("mps")

baselines = {}
for name, cls in [("FNO",          FNO),
                  ("TFNO",         TFNO),
                  ("UNetClassic",  UNetClassic),
                  ("UNetConvNext", UNetConvNext)]:
    model = cls.from_pretrained(f"polymathic-ai/{name}-{dname}").eval()
    if name in SPECTRAL_MODELS:
        model = CPUForwardWrapper(model)
    else:
        model = model.to(device)
    baselines[name] = model

In [ ]:
"""N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)   # which trajectories to evaluate 
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0]     # autoregressive steps after the seed window
N_STEPS_INPUT = 4                                              # N_STEPS_INPUT sliding-window width (must match training)

results = {name: [] for name in list(baselines.keys())}
ground_truth = []

IS_RESIDUAL = {
    "FNO":  True,
    "TFNO": True,
    "UNetClassic": False,
    "UNetConvNext": False,
}

# Safe range based on your diagnostic: seed window was [-1.49, 2.50]
# 3.0 gives comfortable headroom without being too tight
NORM_CLAMP = {
    "FNO":          None, #(-3.5, 3.5),
    "TFNO":         None, #(-3.5, 3.5),
    "UNetClassic":  None,           # no clamping for UNets
    "UNetConvNext": None,
}


for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item      = dataset[traj_idx]
    x_full, _ = get_well_inputs(item, device=device)
    ground_truth.append(x_full)                           # list of (T, H, W, C)
    H, W, C = x_full.shape[1:]  # (T, H, W, C)

    # ════════════════════════════════════════════════════════════════════════
    # BASELINES — single deterministic prediction per step
    # ════════════════════════════════════════════════════════════════════════
    for model_name, model in baselines.items():
        print(f"\n  [{model_name}]")
        model.eval()
        model.to(device)

        seed_raw    = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)
        window_norm = normalizer.normalize_flattened(seed_raw, mode='variable')             # (1, N_STEPS_INPUT, H, W, C)
        bl_preds = torch.zeros(ROLLOUT_STEPS, H, W, C)
        clamp_range = NORM_CLAMP[model_name]

        with torch.no_grad():
            for step in range(ROLLOUT_STEPS):
                gt_idx = N_STEPS_INPUT + step
                if gt_idx >= len(x_full):
                    break

                x_in      = rearrange(window_norm, 'B T H W C -> B (T C) H W')
                yhat      = model(x_in)                                                     # (B, C_out, H, W)
                
                if IS_RESIDUAL[model_name]:
                    # FNO predicts Δx — add back the last frame in normalized space
                    last_hw = rearrange(window_norm[:, -1:], 'B T H W C -> B (T C) H W')
                    yhat    = last_hw + yhat                     # (B, C, H, W)
                
                yhat_norm = rearrange(yhat, 'B C H W -> B 1 H W C')

                # Clamp in normalized space to prevent spectral mode blow-up (FNO only)
                if clamp_range is not None:
                    yhat_norm = yhat_norm.clamp(*clamp_range)

                pred_phys = normalizer.denormalize_flattened(yhat_norm, mode='variable')    # (1, 1, H, W, C)

                #bl_preds[step] = pred_phys.cpu().squeeze(0).squeeze(0)
                bl_preds[step] = pred_phys.squeeze(0).squeeze(0)

                #next_frame_norm = normalizer.normalize_flattened(pred_phys, mode='variable')
                #window_norm = torch.cat([window_norm[:, 1:], next_frame_norm.detach()], dim=1)

                window_norm = torch.cat(
                    [window_norm[:, 1:], yhat_norm.detach()], dim=1)

                print(f"    step {step+1:>4d}/{ROLLOUT_STEPS-N_STEPS_INPUT}", end='\r')

        print()
        results[model_name].append(bl_preds)                            # list of (T, H, W, C)"""

In [ ]:
N_TRAJ        = len(dataset.metadata.n_steps_per_trajectory)   # which trajectories to evaluate 
ROLLOUT_STEPS = dataset.metadata.n_steps_per_trajectory[0]     # autoregressive steps after the seed window
N_STEPS_INPUT = 4                                              # N_STEPS_INPUT sliding-window width (must match training)

results = {name: [] for name in list(baselines.keys())}
ground_truth = []

for traj_idx in range(N_TRAJ):
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item      = dataset[traj_idx]
    x_full, _ = get_well_inputs(item, device=device)
    ground_truth.append(x_full)                           # list of (T, H, W, C)
    H, W, C = x_full.shape[1:]  # (T, H, W, C)

    # ════════════════════════════════════════════════════════════════════════
    # BASELINES — single deterministic prediction per step
    # ════════════════════════════════════════════════════════════════════════
    for model_name, model in baselines.items():
        print(f"\n  [{model_name}]")
        model.eval()
        model.to(device)

        #seed_raw    = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)
        #window_norm = normalizer.normalize_flattened(seed_raw, mode='variable')             # (1, N_STEPS_INPUT, H, W, C)
        
        # seed window in PHYSICAL space — re-normalize each step (mirrors official trainer)
        window_phys = x_full[:N_STEPS_INPUT].unsqueeze(0).to(device)       # (1,N_IN,H,W,C)

        bl_preds = torch.zeros(ROLLOUT_STEPS, H, W, C)


        with torch.no_grad():
            for step in range(ROLLOUT_STEPS):
                gt_idx = N_STEPS_INPUT + step
                if gt_idx >= len(x_full):
                    break

                # 1. normalize window fresh from physical space each step
                window_norm = normalizer.normalize_flattened(
                    window_phys, mode='variable'
                )  # (1, N_STEPS_INPUT, H, W, C)

                # 2. flatten time into channels: (1, N_IN*C, H, W)
                x_in = rearrange(window_norm, 'B T H W C -> B (T C) H W')
                
                # 3. model forward — always predicts full normalized next field
                yhat = model(x_in)                              # (B, C_out, H, W) — full field in norm space

                # 4. reshape to (1,1,H,W,C) then denormalize to physical space
                yhat_phys = normalizer.denormalize_flattened(
                    rearrange(yhat, 'B C H W -> B 1 H W C'),
                    mode='variable'
                )     

                bl_preds[step] = yhat_phys.squeeze(0).squeeze(0).cpu()
                
                # 5. slide window in PHYSICAL space
                window_phys = torch.cat(
                    [window_phys[:, 1:], yhat_phys.detach()], dim=1
                )

                print(f"    step {step+1:>4d}/{ROLLOUT_STEPS-N_STEPS_INPUT}", end='\r')

        print()
        results[model_name].append(bl_preds)                            # list of (T, H, W, C)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [FNO]
    step  197/197

  [TFNO]
    step  197/197

  [UNetClassic]
    step  197/197

  [UNetConvNext]
    step  197/197

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [FNO]
    step  197/197

  [TFNO]
    step  197/197

  [UNetClassic]
    step  197/197

  [UNetConvNext]
    step  197/197

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [FNO]
    step  197/197

  [TFNO]
    step  197/197

  [UNetClassic]
    step  197/197

  [UNetConvNext]
    step  197/197

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [FNO]
    step  197/197

  [TFNO]
    step  197/197

  [UNetClassic

# Save baselines to binary

In [ ]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    results[model_name] = torch.stack(results[model_name], dim = 0).cpu()     # (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
    save_path = save_dir / f"{model_name}_{well_dataset_name}_rollout_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(results[model_name], f)

# Save Ground-truth to binary

In [1]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

for model_name in baselines.keys():
    gt = torch.stack(ground_truth, dim = 0).cpu()     # (N_TRAJ, N_STEPS, H, W, C)
    save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(gt, f)

NameError: name 'Path' is not defined